# Fit action planning model

September 2025

In [1]:
from memo import memo
import jax
import jax.numpy as jnp

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

## Load data

In [31]:
scenario_labels = [
    "basketball",
    "birthday",
    "brunch",
    "conference",
    "cooking",
    "crabs",
    "dip",
    "drinks",
    "driving",
    "fair",
    "gala",
    "hike",
    "oysters",
    "social",
    "soup",
    "wedding",
]

### Load and fill risk, effort, discomfort

In [48]:
risk_summary = pd.read_csv("../data/risk/risk_summary.csv")
risk_summary.insert(
    risk_summary.columns.get_loc("scenario_label") + 1,
    "scenario_idx",
    risk_summary["scenario_label"].apply(lambda x: scenario_labels.index(x))
)
risk_matrix = risk_summary.pivot(index="scenario_idx", columns="action", values="empirical_stat").fillna(0).values
assert risk_matrix.shape == (16, 4)

In [49]:
effort_summary = pd.read_csv("../data/effort/effort_summary.csv")
effort_summary.insert(
    effort_summary.columns.get_loc("scenario_label") + 1,
    "scenario_idx",
    effort_summary["scenario_label"].apply(lambda x: scenario_labels.index(x))
)
effort_summary.head()

effort_matrix = effort_summary.pivot(index="scenario_idx", columns="action", values="empirical_stat").fillna(0).values
assert effort_matrix.shape == (16, 4)

In [ ]:
discomfort_summary = pd.read_csv("../data/discomfort/discomfort_summary.csv")
discomfort_summary.insert(
    discomfort_summary.columns.get_loc("scenario_label") + 1,
    "scenario_idx",
    discomfort_summary["scenario_label"].apply(lambda x: scenario_labels.index(x))
)

discomfort_pivot = discomfort_summary.pivot_table(
    index="scenario_idx",
    columns=["action", "closeness"],
    values="empirical_stat",
    fill_value=0,
)

discomfort_matrix = discomfort_pivot.values.reshape(16, 4, 4) # scenario_idx x action x closeness

,scenario_label,scenario_idx,closeness_condition,closeness,action,n,empirical_stat,ci_lower,mean,ci_upper
0,basketball,0,not_close,0,0,6,1.333333,1.00,1.333324,1.800000
1,basketball,0,not_close,0,1,6,1.333333,1.00,1.336769,2.144643
2,basketball,0,not_close,0,2,6,4.833333,3.00,4.853473,6.500000
3,basketball,0,not_close,0,3,6,6.500000,5.75,6.499797,7.000000
4,basketball,0,somewhat_close,1,0,7,1.428571,1.00,1.419096,2.500000


Turn all risk matrices into jax arrays

In [59]:
risk_matrix = jnp.array(risk_matrix)
effort_matrix = jnp.array(effort_matrix)
discomfort_matrix = jnp.array(discomfort_matrix)

### Load action planning data

In [60]:
data = pd.read_csv("../data/planning-1/main_trials_tidy.csv")
data.insert(
    data.columns.get_loc("scenario_label") + 2,
    "scenario_idx",
    data["scenario_label"].apply(lambda x: scenario_labels.index(x))
)
data.head()

,subject_id,scenario_label,closeness_condition,scenario_idx,closeness,action,likert_rating,p_action
0,4e70ab6c-08df-50a1-bce0-6269c2996b70,crabs,close,5,2,0,3,0.187500
1,4e70ab6c-08df-50a1-bce0-6269c2996b70,crabs,close,5,2,1,6,0.375000
2,4e70ab6c-08df-50a1-bce0-6269c2996b70,crabs,close,5,2,2,3,0.187500
3,4e70ab6c-08df-50a1-bce0-6269c2996b70,crabs,close,5,2,3,4,0.250000
4,4e70ab6c-08df-50a1-bce0-6269c2996b70,cooking,somewhat_close,4,1,0,2,0.142857


## Model

Base model (H0): $p(a|c) \propto \exp(\alpha \cdot (-w_r \cdot c_{\text{risk}}(a) - w_e \cdot c_{\text{effort}}(a)))$

Relationship model (H1): $p(a|c) \propto \exp(\alpha \cdot (-w_d \cdot c_{\text{discomfort}}(a|c) - w_e \cdot c_{\text{effort}}(a)))$

In [61]:
risk_levels = [0, 1, 2, 3] 
closeness_levels = [0, 1, 2, 3] 

In [62]:
@jax.jit
def c_risk(scenario_idx, a):
    return risk_matrix[scenario_idx, a]

@jax.jit
def c_effort(scenario_idx, a):
    return effort_matrix[scenario_idx, a]

@jax.jit
def c_discomfort(scenario_idx, a, c):
    return discomfort_matrix[scenario_idx, a, c]

In [63]:
@memo
def actor_h0[a: risk_levels, c: closeness_levels](scenario_idx, alpha, w_r, w_e):
    cast: [actor]
    actor: chooses(
        a in risk_levels,
        wpp=exp(
            alpha * (-w_r * c_risk(scenario_idx, a) - w_e * c_effort(scenario_idx, a))
        ),
    )
    return Pr[actor.a == a]


@memo
def actor_h1[a: risk_levels, c: closeness_levels](scenario_idx, alpha, w_d, w_e):
    cast: [actor]
    actor: knows(c)
    actor: chooses(
        a in risk_levels,
        wpp=exp(
            alpha
            * (
                -w_d * c_discomfort(scenario_idx, a, c)
                - w_e * c_effort(scenario_idx, a)
            )
        ),
    )
    return Pr[actor.a == a]

In [64]:
actor_h0(0, 1, 1, 1, print_table=True)

+----------------+---------------------+-----------------------+
| a: risk_levels | c: closeness_levels | actor_h0              |
+----------------+---------------------+-----------------------+
| 0              | 0                   | 0.7818808555603027    |
| 0              | 1                   | 0.7818808555603027    |
| 0              | 2                   | 0.7818808555603027    |
| 0              | 3                   | 0.7818808555603027    |
| 1              | 0                   | 0.16412732005119324   |
| 1              | 1                   | 0.16412732005119324   |
| 1              | 2                   | 0.16412732005119324   |
| 1              | 3                   | 0.16412732005119324   |
| 2              | 0                   | 0.04707746580243111   |
| 2              | 1                   | 0.04707746580243111   |
| 2              | 2                   | 0.04707746580243111   |
| 2              | 3                   | 0.04707746580243111   |
| 3              | 0     

Array([[0.78188086],
       [0.16412732],
       [0.04707747],
       [0.0069143 ]], dtype=float32)

In [65]:
actor_h1(0, 1, 1, 1, print_table=True)

+----------------+---------------------+------------------------+
| a: risk_levels | c: closeness_levels | actor_h1               |
+----------------+---------------------+------------------------+
| 0              | 0                   | 0.8036112189292908     |
| 0              | 1                   | 0.837166965007782      |
| 0              | 2                   | 0.8565000891685486     |
| 0              | 3                   | 0.7476088404655457     |
| 1              | 0                   | 0.1858295202255249     |
| 1              | 1                   | 0.10932325571775436    |
| 1              | 2                   | 0.0814247876405716     |
| 1              | 3                   | 0.19589799642562866    |
| 2              | 0                   | 0.00861410889774561    |
| 2              | 1                   | 0.04021777585148811    |
| 2              | 2                   | 0.03681955486536026    |
| 2              | 3                   | 0.04069744423031807    |
| 3       

Array([[0.8036112 , 0.83716697, 0.8565001 , 0.74760884],
       [0.18582952, 0.10932326, 0.08142479, 0.195898  ],
       [0.00861411, 0.04021778, 0.03681955, 0.04069744],
       [0.00194509, 0.01329205, 0.02525553, 0.01579569]], dtype=float32)